In [1]:
!pip install streamlit pyngrok python-jobspy langchain langchain-google-genai pdfminer.six -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 19.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but yo

In [2]:
import os
from google.colab import userdata
os.environ['GEMINI']=userdata.get('GEMINI')

In [3]:
%%writefile pages/resume_reader.py

import streamlit as st
from pdfminer.high_level import extract_text
import re
import sqlite3

# =========================================
# PAGE CONFIG
# =========================================

st.set_page_config(
    page_title="Resume Analyser",
    page_icon="📄",
    layout="wide"
)

# =========================================
# DATABASE HELPERS
# =========================================

DB_PATH = "jobify_user.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""
        CREATE TABLE IF NOT EXISTS user_profile (
            id INTEGER PRIMARY KEY,
            full_name TEXT,
            email TEXT,
            phone TEXT,
            desired_job_title TEXT,
            preferred_location TEXT,
            preferred_sites TEXT,
            experience_level TEXT,
            skills TEXT,
            preferred_industry TEXT,
            hours_old INTEGER
        )
    """)
    conn.commit()
    conn.close()

def save_resume_to_profile(name, email, phone, skills):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT id FROM user_profile ORDER BY id DESC LIMIT 1")
    row = c.fetchone()
    skills_str = ", ".join(skills) if skills else ""
    if row:
        c.execute("""
            UPDATE user_profile
            SET full_name=?, email=?, phone=?, skills=?
            WHERE id=?
        """, (name, email, phone, skills_str, row[0]))
    else:
        c.execute("""
            INSERT INTO user_profile (full_name, email, phone, skills, hours_old)
            VALUES (?, ?, ?, ?, 72)
        """, (name, email, phone, skills_str))
    conn.commit()
    conn.close()

init_db()

# =========================================
# HEADER
# =========================================

st.markdown("""
<div style="
    background: linear-gradient(to right, #141e30, #243b55);
    padding: 30px;
    border-radius: 15px;
    color: white;
    text-align: center;
">
    <h1>AI Resume Analyser</h1>
    <p>Upload your resume and let our AI analyse it for you</p>
</div>
""", unsafe_allow_html=True)

st.write("")

# =========================================
# FILE UPLOAD
# =========================================

uploaded_file = st.file_uploader(
    "Upload Resume as PDF",
    type="pdf"
)

# =========================================
# PDF TEXT EXTRACTION
# =========================================

def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)

# =========================================
# CLEAN TEXT
# =========================================

def clean_text(text):

    text = re.sub(r'\t', ' ', text)
    text = re.sub(r'\r', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    sections = [
        "PROFESSIONAL SUMMARY",
        "SKILLS",
        "WORK EXPERIENCE",
        "EDUCATION",
        "PROJECTS",
        "CERTIFICATIONS",
        "ACHIEVEMENTS",
        "LANGUAGES",
        "INTERESTS"
    ]

    for section in sections:
        text = text.replace(section, f"\n\n{section}\n")

    return text

# =========================================
# EXTRACT EMAIL
# =========================================

def extract_email(text):

    email = re.search(r'\S+@\S+', text)

    return email.group(0) if email else "Not Found"

# =========================================
# EXTRACT PHONE NUMBER
# =========================================

def extract_phone_number(text):

    phone_regex = r'(?:\+?\d{1,3}[-.\s]?)?\(?\d{2,4}\)?[-.\s]?\d{3,4}[-.\s]?\d{3,4}(?:[-.\s]?\d{1,4})?'

    match = re.search(phone_regex, text)

    return match.group(0) if match else "Not Found"

# =========================================
# EXTRACT NAME
# =========================================

def extract_name(text):

    lines = text.split()

    possible_name = []

    for word in lines[:10]:

        if word.isalpha() and word[0].isupper():
            possible_name.append(word)

    if len(possible_name) >= 2:
        return possible_name[0] + " " + possible_name[1]

    elif len(possible_name) == 1:
        return possible_name[0]

    return "Not Found"

# =========================================
# EXTRACT SKILLS
# =========================================

def extract_skills(text):

    skills_section = re.search(
        r'SKILLS(.*?)(WORK EXPERIENCE|EDUCATION|PROJECTS|CERTIFICATIONS|ACHIEVEMENTS|LANGUAGES|INTERESTS|PROFILE|SUMMARY)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    if not skills_section:
        return []

    skills_text = skills_section.group(1)
    skills_text = re.sub(r'\s+', ' ', skills_text)
    raw_skills = re.split(r',|•|\|/|\n', skills_text)

    skills = []

    for skill in raw_skills:

        skill = skill.strip()

        if ":" in skill:
            parts = skill.split(":")
            if len(parts) > 1:
                skill = parts[1].strip()

        if len(skill) < 2:
            continue

        blocked_words = [
            "skills", "work experience", "education",
            "projects", "certifications", "languages", "interests"
        ]

        if skill.lower() in blocked_words:
            continue

        skills.append(skill)

    skills = list(dict.fromkeys(skills))

    return skills

# =========================================
# EXTRACT EDUCATION
# =========================================

def extract_education(text):

    education_section = re.search(
        r'EDUCATION(.*?)(PROJECTS|CERTIFICATIONS|ACHIEVEMENTS|LANGUAGES|INTERESTS)',
        text,
        re.IGNORECASE
    )

    if education_section:
        return education_section.group(1).strip()

    return "Not Found"

# =========================================
# EXTRACT WORK EXPERIENCE
# =========================================

def extract_work_experience(text):

    work_section = re.search(
        r'WORK EXPERIENCE(.*?)(EDUCATION|PROJECTS|CERTIFICATIONS|ACHIEVEMENTS|LANGUAGES|INTERESTS)',
        text,
        re.IGNORECASE
    )

    if work_section:
        return work_section.group(1).strip()

    return "Not Found"

# =========================================
# MAIN PARSER
# =========================================

def parse():

    sample_text = extract_text_from_pdf("temp.pdf")

    cleaned_text = clean_text(sample_text)

    name = extract_name(cleaned_text)

    email = extract_email(cleaned_text)

    phone_number = extract_phone_number(cleaned_text)

    skills = extract_skills(cleaned_text)

    education = extract_education(cleaned_text)

    work_experience = extract_work_experience(cleaned_text)

    # =====================================
    # SUCCESS MESSAGE
    # =====================================

    st.success("Resume analysed successfully!")

    st.write("")

    # =====================================
    # PROFILE CARD
    # =====================================

    st.markdown(f"""
    <div style="
        background-color:#1e1e1e;
        padding:30px;
        border-radius:15px;
        color:white;
        text-align:center;
    ">
        <h1>{name}</h1>
        <p>Resume Analysis Dashboard</p>
    </div>
    """, unsafe_allow_html=True)

    st.write("")

    # =====================================
    # SAVE TO PROFILE BUTTON
    # =====================================

    st.info("💾 Want to use these details for job search? Save them to your profile!")

    if "saved" not in st.session_state:
      st.session_state.saved = False

    if st.button("💾 Save Resume Details to My Profile", use_container_width=True, type="primary"):
      save_resume_to_profile(name, email, phone_number, skills)
      st.session_state.saved = True
      st.success("Details saved to your profile!")

    if st.session_state.saved:
      st.page_link("pages/pref.py", label="→ Go to Your Preferences")

    # =====================================
    # CONTACT INFO
    # =====================================

    st.subheader("Contact Information")

    col1, col2 = st.columns(2)

    with col1:

        st.markdown(f"""
        <div style="
            background-color:#262730;
            padding:20px;
            border-radius:12px;
            color:white;
        ">
            <h3>Email</h3>
            <p>{email}</p>
        </div>
        """, unsafe_allow_html=True)

    with col2:

        st.markdown(f"""
        <div style="
            background-color:#262730;
            padding:20px;
            border-radius:12px;
            color:white;
        ">
            <h3>Phone Number</h3>
            <p>{phone_number}</p>
        </div>
        """, unsafe_allow_html=True)

    st.write("")

    # =====================================
    # SKILLS SECTION
    # =====================================

    st.subheader("Skills")

    if skills:

        skills_html = ""

        for skill in skills:

            skills_html += f"""
            <span style="
                display:inline-block;
                background-color:#4CAF50;
                color:white;
                padding:10px 18px;
                margin:6px;
                border-radius:20px;
                font-size:14px;
            ">
                {skill}
            </span>
            """

        st.markdown(skills_html, unsafe_allow_html=True)

    else:

        st.warning("No skills detected")

    st.write("")

    # =====================================
    # FULL RESUME TEXT
    # =====================================

    with st.expander("View Full Extracted Resume Text"):

        st.markdown(f"""
        <div style="
            background-color:#f4f4f4;
            padding:20px;
            border-radius:10px;
            color:black;
        ">
            {cleaned_text}
        </div>
        """, unsafe_allow_html=True)

    return name, email, phone_number, skills

# =========================================
# FILE HANDLING
# =========================================

if uploaded_file is not None:

    with open("temp.pdf", "wb") as f:

        f.write(uploaded_file.read())

    parse()

Writing pages/resume_reader.py


In [4]:
%%writefile pages/pref.py

import streamlit as st
import sqlite3

# =========================================
# PAGE CONFIG
# =========================================

st.set_page_config(
    page_title="Your Preferences",
    page_icon="⚙️",
    layout="wide"
)

# =========================================
# DATABASE
# =========================================

DB_PATH = "jobify_user.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""
        CREATE TABLE IF NOT EXISTS user_profile (
            id INTEGER PRIMARY KEY,
            full_name TEXT,
            email TEXT,
            phone TEXT,
            desired_job_title TEXT,
            preferred_location TEXT,
            preferred_sites TEXT,
            experience_level TEXT,
            skills TEXT,
            preferred_industry TEXT,
            hours_old INTEGER
        )
    """)
    conn.commit()
    conn.close()

def save_profile(data):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("DELETE FROM user_profile")
    c.execute("""
        INSERT INTO user_profile (
            full_name, email, phone, desired_job_title, preferred_location,
            preferred_sites, experience_level, skills, preferred_industry, hours_old
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        data["full_name"], data["email"], data["phone"],
        data["desired_job_title"], data["preferred_location"],
        data["preferred_sites"], data["experience_level"],
        data["skills"], data["preferred_industry"], data["hours_old"]
    ))
    conn.commit()
    conn.close()

def load_profile():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT * FROM user_profile ORDER BY id DESC LIMIT 1")
    row = c.fetchone()
    conn.close()

    if row:
        keys = [
            "id", "full_name", "email", "phone", "desired_job_title",
            "preferred_location", "preferred_sites", "experience_level",
            "skills", "preferred_industry", "hours_old"
        ]
        return dict(zip(keys, row))

    return {}

def clear_profile():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("DELETE FROM user_profile")
    conn.commit()
    conn.close()

init_db()
existing = load_profile()

# =========================================
# HEADER
# =========================================

st.markdown("""
<div style="
    background: linear-gradient(to right, #141e30, #243b55);
    padding: 30px;
    border-radius: 15px;
    color: white;
    text-align: center;
">
    <h1>⚙️ Your Preferences</h1>
    <p>Set your job search preferences for personalised results</p>
</div>
""", unsafe_allow_html=True)

st.write("")
if st.button("📄 Save details from Resume instead", use_container_width=True):
    st.switch_page("pages/res.py")
# =========================================
# FORM
# =========================================


col1, col2 = st.columns(2)

with col1:
    full_name = st.text_input("Full Name", value=existing.get("full_name", ""))
    email = st.text_input("Email Address", value=existing.get("email", ""))
    phone = st.text_input("Phone Number", value=existing.get("phone", ""))
    desired_job_title = st.text_input(
        "Desired Job Title",
        value=existing.get("desired_job_title", ""),
        placeholder="e.g. Python Developer, Data Analyst"
    )
    preferred_location = st.text_input(
        "Preferred Location",
        value=existing.get("preferred_location", ""),
        placeholder="e.g. Bengaluru, India"
    )

with col2:
    preferred_industry = st.text_input(
        "Preferred Industry",
        value=existing.get("preferred_industry", ""),
        placeholder="e.g. Software, Finance, Healthcare"
    )

    experience_options = ["Entry Level", "Mid Level", "Senior Level", "Lead / Manager", "Executive"]
    current_exp = existing.get("experience_level", "Entry Level")

    exp_index = experience_options.index(current_exp) if current_exp in experience_options else 0

    experience_level = st.selectbox(
        "Experience Level",
        experience_options,
        index=exp_index
    )

    all_sites = ["indeed", "linkedin", "glassdoor", "naukri", "google"]
    saved_sites = existing.get("preferred_sites", "")
    default_sites = [s for s in saved_sites.split(",") if s in all_sites] if saved_sites else ["linkedin", "naukri"]

    preferred_sites = st.multiselect(
        "Preferred Job Sites",
        options=all_sites,
        default=default_sites
    )

    hours_old = st.slider(
        "Show jobs posted in the last (hours)",
        min_value=24, max_value=720, step=24,
        value=int(existing.get("hours_old", 72))
    )

    skills = st.text_area(
        "Your Key Skills (comma separated)",
        value=existing.get("skills", ""),
        placeholder="e.g. Python, SQL, Machine Learning",
        height=105
    )

st.write("")

# =========================================
# ACTION BUTTONS
# =========================================

col_a, col_b = st.columns(2)

with col_a:
    if st.button("💾 Save Preferences", use_container_width=True, type="primary"):
        if not full_name or not desired_job_title:
            st.error("Please fill in at least your Full Name and Desired Job Title.")
        else:
            save_profile({
                "full_name": full_name,
                "email": email,
                "phone": phone,
                "desired_job_title": desired_job_title,
                "preferred_location": preferred_location,
                "preferred_sites": ",".join(preferred_sites),
                "experience_level": experience_level,
                "skills": skills,
                "preferred_industry": preferred_industry,
                "hours_old": hours_old
            })
            st.success("Preferences saved!")
            st.rerun()

with col_b:
    if st.button("🗑️ Clear Saved Data", use_container_width=True):
        clear_profile()
        st.success("Saved data cleared!")
        st.rerun()

# =========================================
# SAVED PROFILE PREVIEW
# =========================================

saved = load_profile()

if saved:
    st.write("")
    st.divider()
    st.subheader("📋 Saved Profile")

    c1, c2 = st.columns(2)

    with c1:
        st.markdown(f"**Name:** {saved.get('full_name', '-')}")
        st.markdown(f"**Email:** {saved.get('email', '-')}")
        st.markdown(f"**Phone:** {saved.get('phone', '-')}")
        st.markdown(f"**Desired Role:** {saved.get('desired_job_title', '-')}")
        st.markdown(f"**Location:** {saved.get('preferred_location', '-')}")

    with c2:
        st.markdown(f"**Industry:** {saved.get('preferred_industry', '-')}")
        st.markdown(f"**Experience:** {saved.get('experience_level', '-')}")
        st.markdown(f"**Sites:** {saved.get('preferred_sites', '-')}")
        st.markdown(f"**Jobs from last:** {saved.get('hours_old', '-')} hours")
        st.markdown(f"**Skills:** {saved.get('skills', '-')}")

Writing pages/pref.py


In [5]:
%%writefile pages/ai_job_search.py

import streamlit as st
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from typing import Optional, List
from jobspy import scrape_jobs
import pandas as pd
import sqlite3
import traceback
import os


def _clean_html(html: str) -> str:
    """Collapse a multi-line HTML string into a single line.

    Streamlit's markdown renderer treats any line indented by 4+ spaces as a
    code block, which causes raw <div>/<br> tags to show up as literal text
    when an f-string keeps Python's indentation. Stripping each line and
    joining removes that whitespace so the HTML renders correctly.
    """
    return "".join(line.strip() for line in html.strip().splitlines())

# =========================================
# API KEY
# (Left as-is per project setup. For a production app this should live in
# st.secrets / an environment variable instead of being hardcoded.)
# =========================================

gem = os.environ.get('GEMINI')
if not gem:
    st.error("Gemini API key not found. Set GEMINI_API_KEY before launching the app.")
    st.stop()

# =========================================
# PAGE CONFIG
# =========================================

st.set_page_config(
    page_title="AI Job Search",
    page_icon="💼",
    layout="wide"
)

# =========================================
# DATABASE HELPER
# =========================================

DB_PATH = "jobify_user.db"

def load_profile() -> dict:
    """Load the most recent saved user profile. Returns {} if anything goes
    wrong (missing DB, missing table, empty table, etc.) so the rest of the
    app can degrade gracefully."""
    try:
        conn = sqlite3.connect(DB_PATH)
        c = conn.cursor()
        c.execute("SELECT * FROM user_profile ORDER BY id DESC LIMIT 1")
        row = c.fetchone()
        conn.close()
        if row:
            keys = ["id", "full_name", "email", "phone", "desired_job_title",
                    "preferred_location", "preferred_sites", "experience_level",
                    "skills", "preferred_industry", "hours_old"]
            return dict(zip(keys, row))
    except Exception:
        pass
    return {}

# =========================================
# CUSTOM CSS
# =========================================

st.markdown("""
<style>

.main {
    background-color: #0e1117;
}

.job-card {
    background-color: #1e1e1e;
    padding: 22px;
    border-radius: 15px;
    margin-bottom: 22px;
    border: 1px solid #333;
}

.job-title {
    color: #4CAF50;
    font-size: 24px;
    font-weight: bold;
}

.company {
    color: white;
    font-size: 18px;
    font-weight: 600;
}

.location {
    color: #cccccc;
    font-size: 15px;
}

.salary {
    color: #00e676;
    font-size: 16px;
    font-weight: bold;
}

.section-title {
    color: white;
    font-size: 15px;
    margin-top: 6px;
}

.apply-button {
    background-color: #4CAF50;
    color: white;
    padding: 10px 18px;
    border-radius: 10px;
    text-decoration: none;
}

.profile-banner {
    background: linear-gradient(to right, #1a3a1a, #243b55);
    padding: 20px;
    border-radius: 12px;
    color: white;
    margin-bottom: 20px;
    border: 1px solid #4CAF50;
}

.badge {
    display: inline-block;
    background-color: #2c2f33;
    color: #e0e0e0;
    border: 1px solid #4CAF50;
    border-radius: 999px;
    padding: 3px 12px;
    margin: 3px 6px 3px 0;
    font-size: 12px;
}

.company-box {
    background-color: #16181c;
    border: 1px solid #333;
    border-radius: 10px;
    padding: 12px 14px;
    margin-top: 10px;
}

.company-box-title {
    color: #4CAF50;
    font-weight: 600;
    margin-bottom: 6px;
}

.muted {
    color: #999999;
    font-size: 13px;
}

</style>
""", unsafe_allow_html=True)

# =========================================
# HEADER
# =========================================

st.markdown("""
<div style="
    background: linear-gradient(to right, #141e30, #243b55);
    padding: 35px;
    border-radius: 15px;
    text-align: center;
    color: white;
">
    <h1>AI Job Search</h1>
    <p>Find jobs intelligently using AI — with richer company &amp; role details</p>
</div>
""", unsafe_allow_html=True)

st.write("")

# =========================================
# LLM
# =========================================

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=gem
)

# =========================================
# STRUCTURED OUTPUT
# =========================================

class JobsInfo(BaseModel):

    site: Optional[List[str]] = Field(
        description="The job site(s) to search for jobs on, e.g. linkedin, indeed, naukri, glassdoor, google"
    )

    search_term: str = Field(
        description="The job title / role to search for"
    )

    google_search_term: str = Field(
        description="A natural-language search string for Google Jobs that includes the role, location and time filter"
    )

    location: Optional[str] = Field(
        description="The location to search for jobs in"
    )

    results_wanted: Optional[int] = Field(
        description="The number of results to return per site"
    )

    hours_old: Optional[int] = Field(
        description="The maximum age (in hours) of job postings to return"
    )

    country_indeed: Optional[str] = Field(
        description="The country to use for Indeed/Glassdoor searches"
    )

    is_remote: Optional[bool] = Field(
        default=None,
        description="Whether the user is specifically looking for remote jobs"
    )

    job_type: Optional[str] = Field(
        default=None,
        description="The type of job requested, e.g. fulltime, parttime, internship, contract"
    )

# =========================================
# AGENT
# =========================================

agent = create_agent(
    model=llm,
    response_format=JobsInfo
)

# =========================================
# HELPERS
# =========================================

def _safe_get(row, key, default=""):
    """Get a value from a pandas row, treating NaN/None as the default.

    String values have embedded newlines/carriage returns collapsed to
    spaces so they don't break single-line HTML rendering later.
    """
    val = row.get(key, default)
    if val is None:
        return default
    try:
        if pd.isna(val):
            return default
    except (TypeError, ValueError):
        pass
    if isinstance(val, str):
        val = " ".join(val.splitlines())
    return val


def _format_salary(row) -> str:
    """Build a readable salary string from jobspy's salary fields."""
    min_amt = _safe_get(row, "min_amount", None)
    max_amt = _safe_get(row, "max_amount", None)
    currency = _safe_get(row, "currency", "")
    interval = _safe_get(row, "interval", "")

    if not min_amt and not max_amt:
        return ""

    def _fmt(num):
        try:
            return f"{float(num):,.0f}"
        except (TypeError, ValueError):
            return str(num)

    symbol = "₹" if currency in ("", "INR") else f"{currency} "

    if min_amt and max_amt and min_amt != max_amt:
        text = f"{symbol}{_fmt(min_amt)} - {symbol}{_fmt(max_amt)}"
    else:
        amt = max_amt or min_amt
        text = f"{symbol}{_fmt(amt)}"

    if interval:
        text += f" / {interval}"

    return text


def _format_badges(row) -> str:
    """Build a row of small badges for job type, remote status, level, etc."""
    badges = []

    job_type = _safe_get(row, "job_type", "")
    if job_type:
        for jt in str(job_type).replace("[", "").replace("]", "").replace("'", "").split(","):
            jt = jt.strip()
            if jt:
                badges.append(jt.replace("_", " ").title())

    if _safe_get(row, "is_remote", False) in (True, "True", "true", 1):
        badges.append("Remote")

    work_type = _safe_get(row, "work_from_home_type", "")
    if work_type:
        badges.append(str(work_type).replace("_", " ").title())

    job_level = _safe_get(row, "job_level", "")
    if job_level:
        badges.append(str(job_level).replace("_", " ").title())

    job_function = _safe_get(row, "job_function", "")
    if job_function:
        badges.append(str(job_function).replace("_", " ").title())

    if not badges:
        return ""

    return "".join(f'<span class="badge">{b}</span>' for b in badges)


def _format_company_box(row) -> str:
    """Build an HTML block with extra company / employer details."""
    company_industry = _safe_get(row, "company_industry", "")
    company_employees = _safe_get(row, "company_num_employees", "")
    company_revenue = _safe_get(row, "company_revenue", "")
    company_url = _safe_get(row, "company_url", "")
    company_addresses = _safe_get(row, "company_addresses", "")

    details = []
    if company_industry:
        details.append(f"🏢 <b>Industry:</b> {company_industry}")
    if company_employees:
        details.append(f"👥 <b>Company size:</b> {company_employees}")
    if company_revenue:
        details.append(f"💰 <b>Revenue:</b> {company_revenue}")
    if company_addresses:
        details.append(f"📌 <b>HQ / Address:</b> {company_addresses}")
    if company_url:
        details.append(f"🔗 <b>Company page:</b> <a href='{company_url}' target='_blank' style='color:#4CAF50;'>{company_url}</a>")

    if not details:
        return ""

    rows_html = "<br>".join(details)
    return _clean_html(f"""
    <div class="company-box">
        <div class="company-box-title">About the company</div>
        <div style="color:#dddddd; font-size:14px; line-height:1.7;">{rows_html}</div>
    </div>
    """)


def _render_job_card(row):

    title = _safe_get(row, "title", "No Title")
    company = _safe_get(row, "company", "Unknown Company")
    location = _safe_get(row, "location", "Unknown Location")
    site = _safe_get(row, "site", "Unknown Site")
    description = _safe_get(row, "description", "")
    job_url = _safe_get(row, "job_url", None)
    job_url_direct = _safe_get(row, "job_url_direct", None)
    date_posted = _safe_get(row, "date_posted", "Unknown")
    emails = _safe_get(row, "emails", "")
    skills = _safe_get(row, "skills", "")
    experience_range = _safe_get(row, "experience_range", "")
    vacancy_count = _safe_get(row, "vacancy_count", "")

    salary_text = _format_salary(row)
    badges_html = _format_badges(row)
    company_box_html = _format_company_box(row)

    short_description = ""
    if isinstance(description, str) and description:
        short_description = description[:400] + ("..." if len(description) > 400 else "")
    else:
        short_description = "No description available."

    extra_meta = []
    if experience_range:
        extra_meta.append(f"📈 <b>Experience:</b> {experience_range}")
    if vacancy_count:
        extra_meta.append(f"🧾 <b>Openings:</b> {vacancy_count}")
    if skills:
        extra_meta.append(f"🛠️ <b>Skills:</b> {skills}")
    if emails:
        extra_meta.append(f"✉️ <b>Contact:</b> {emails}")
    extra_meta_html = "<br>".join(extra_meta)

    st.markdown(_clean_html(f"""
    <div class="job-card">
        <div class="job-title">{title}</div>
        <div class="company">🏢 {company}</div>
        <div class="location">📍 {location}</div>
        <div class="section-title">🌐 Source: {site} &nbsp;|&nbsp; 🕒 Posted: {date_posted}</div>
        <div style="margin-top:8px;">{badges_html}</div>
        {f'<div class="salary" style="margin-top:8px;">{salary_text}</div>' if salary_text else ''}
        <br>
        <div style="color:#dddddd; line-height:1.6;">{short_description}</div>
        {f'<div class="muted" style="margin-top:8px;">{extra_meta_html}</div>' if extra_meta_html else ''}
        {company_box_html}
        <br>
    """), unsafe_allow_html=True)

    if isinstance(description, str) and len(description) > 400:
        with st.expander("📄 Read full job description"):
            st.write(description)

    col1, col2 = st.columns(2)
    with col1:
        if job_url:
            st.link_button("Apply Now", job_url, use_container_width=True)
    with col2:
        if job_url_direct and job_url_direct != job_url:
            st.link_button("Apply on Company Site", job_url_direct, use_container_width=True)

    st.markdown("</div>", unsafe_allow_html=True)


# jobspy's accepted site identifiers (lowercase), with common aliases /
# variations the LLM agent or the user might type. Used to normalize
# whatever casing/spelling shows up.
KNOWN_SITES = {
    "indeed": "indeed",
    "indeed.com": "indeed",
    "linkedin": "linkedin",
    "linkedin.com": "linkedin",
    "glassdoor": "glassdoor",
    "glassdoor.com": "glassdoor",
    "google": "google",
    "google.com": "google",
    "google jobs": "google",
    "naukri": "naukri",
    "naukri.com": "naukri",
    "zip_recruiter": "zip_recruiter",
    "ziprecruiter": "zip_recruiter",
    "zip recruiter": "zip_recruiter",
    "ziprecruiter.com": "zip_recruiter",
    "bayt": "bayt",
    "bayt.com": "bayt",
    "bdjobs": "bdjobs",
    "bdjobs.com": "bdjobs",
}

# Job sites that people commonly type but that the underlying `jobspy`
# scraper does NOT support. If the user explicitly mentions one of these,
# we let them know it can't be included rather than silently dropping it.
UNSUPPORTED_SITES = [
    "monster", "shine", "internshala", "foundit", "monsterindia",
    "monster india", "timesjobs", "wellfound", "angellist", "dice",
    "simplyhired", "careerbuilder", "stepstone", "totaljobs", "reed",
    "seek", "freshersworld", "instahyre", "cutshort", "hirist",
    "upwork", "freelancer",
]


def _normalize_sites(sites) -> list:
    """Lowercase / map site names to the exact strings jobspy expects, and
    drop anything unrecognized rather than letting it silently break the
    whole scrape."""
    if not sites:
        return []
    if isinstance(sites, str):
        sites = [sites]

    normalized = []
    for s in sites:
        key = str(s).strip().lower()
        mapped = KNOWN_SITES.get(key)
        if mapped and mapped not in normalized:
            normalized.append(mapped)
    return normalized


def _ensure_requested_sites(sites: list, prompt_text: str):
    """Safety net: if the user explicitly typed a supported site name (e.g.
    'naukri') in their prompt but the AI agent didn't include it in the
    structured output, add it back in.

    Also scans for site names jobspy doesn't support (e.g. 'monster') so the
    UI can tell the user those won't be searched.

    Returns (sites, unsupported_mentions).
    """
    sites = list(sites)
    prompt_lower = (prompt_text or "").lower()

    for key, mapped in KNOWN_SITES.items():
        if key in prompt_lower and mapped not in sites:
            sites.append(mapped)

    unsupported_mentions = []
    for name in UNSUPPORTED_SITES:
        if name in prompt_lower and name not in unsupported_mentions:
            unsupported_mentions.append(name)

    return sites, unsupported_mentions


def _run_agent(prompt_text):
    """Call the LLM agent with basic error handling for flaky / Colab
    environments (network issues, rate limits, etc.)."""
    try:
        result = agent.invoke({
            "messages": [{"role": "user", "content": prompt_text}]
        })
        return result["structured_response"], None
    except Exception as e:
        return None, f"AI parsing failed ({e}). Falling back to basic search settings."


def _run_scrape(**kwargs):
    """Wrap scrape_jobs so a failure on one site (or network issue) doesn't
    crash the whole app. Also returns a per-site result count so the UI can
    flag sites (e.g. naukri) that returned zero results."""
    requested_sites = kwargs.get("site_name") or []
    try:
        jobs = scrape_jobs(**kwargs)
        df = pd.DataFrame(jobs)
        site_counts = {s: 0 for s in requested_sites}
        if not df.empty and "site" in df.columns:
            for site_val, count in df["site"].value_counts().items():
                site_counts[str(site_val)] = int(count)
        return df, None, site_counts
    except Exception as e:
        return pd.DataFrame(), f"Job search failed: {e}", {s: 0 for s in requested_sites}


def _apply_filters_and_sort(df: pd.DataFrame) -> pd.DataFrame:
    """Sidebar filters + sorting applied to a results dataframe. Returns the
    filtered/sorted copy. Safe to call on an empty dataframe."""
    if df.empty:
        return df

    filtered = df.copy()

    with st.sidebar:
        st.markdown("### 🔧 Refine Results")

        if "site" in filtered.columns:
            site_options = sorted([s for s in filtered["site"].dropna().unique().tolist()])
            chosen_sites = st.multiselect("Source site", site_options, default=site_options)
            if chosen_sites:
                filtered = filtered[filtered["site"].isin(chosen_sites)]

        if "job_type" in filtered.columns:
            jt_options = sorted({
                str(jt).strip()
                for cell in filtered["job_type"].dropna().tolist()
                for jt in str(cell).replace("[", "").replace("]", "").replace("'", "").split(",")
                if str(jt).strip()
            })
            if jt_options:
                chosen_jt = st.multiselect("Job type", jt_options, default=[])
                if chosen_jt:
                    filtered = filtered[filtered["job_type"].astype(str).apply(
                        lambda v: any(jt in v for jt in chosen_jt)
                    )]

        if "is_remote" in filtered.columns:
            remote_only = st.checkbox("Remote only", value=False)
            if remote_only:
                filtered = filtered[filtered["is_remote"] == True]  # noqa: E712

        if "min_amount" in filtered.columns:
            min_salary = st.number_input(
                "Minimum salary (filters out unspecified salaries)",
                min_value=0, value=0, step=10000
            )
            if min_salary > 0:
                filtered = filtered[filtered["min_amount"].fillna(0) >= min_salary]

        sort_options = {"Most recent": "date_posted", "Highest salary": "max_amount", "Company name": "company"}
        sort_label = st.selectbox("Sort by", list(sort_options.keys()), index=0)
        sort_col = sort_options[sort_label]
        if sort_col in filtered.columns:
            ascending = sort_label == "Company name"
            try:
                filtered = filtered.sort_values(by=sort_col, ascending=ascending, na_position="last")
            except Exception:
                pass

    return filtered


def _render_results(df: pd.DataFrame):
    if df.empty:
        st.warning("No jobs found matching your search/filters. Try widening your search.")
        return

    filtered = _apply_filters_and_sort(df)

    st.success(f"Showing {len(filtered)} of {len(df)} jobs found")

    csv_data = df.to_csv(index=False).encode("utf-8")
    st.download_button(
        "⬇️ Download all results as CSV",
        data=csv_data,
        file_name="job_search_results.csv",
        mime="text/csv"
    )

    st.write("")

    for _, row in filtered.iterrows():
        _render_job_card(row)


# =========================================
# SEARCH FROM PROFILE SECTION
# =========================================

profile = load_profile()

if profile.get("desired_job_title"):

    st.markdown(f"""
    <div class="profile-banner">
        <h4>👤 Profile Detected: {profile.get('full_name', 'User')}</h4>
        <p>
            🎯 <b>Role:</b> {profile.get('desired_job_title', '-')} &nbsp;|&nbsp;
            📍 <b>Location:</b> {profile.get('preferred_location', '-')} &nbsp;|&nbsp;
            🌐 <b>Sites:</b> {profile.get('preferred_sites', '-')}
        </p>
    </div>
    """, unsafe_allow_html=True)

    if st.button("🔍 Search Jobs Using My Profile", use_container_width=True, type="primary"):

        job_title = profile.get("desired_job_title", "developer")
        location = profile.get("preferred_location", "Bengaluru, India")
        sites_str = profile.get("preferred_sites", "linkedin,naukri")
        try:
            hours_old = int(profile.get("hours_old", 72))
        except (TypeError, ValueError):
            hours_old = 72
        sites = [s.strip() for s in sites_str.split(",") if s.strip()] or ["linkedin", "naukri"]

        auto_prompt = (
            f"I want {job_title} jobs in {location} "
            f"from {', '.join(sites)} posted in the last {hours_old} hours"
        )

        st.info(f"🤖 Searching: **{auto_prompt}**")

        with st.spinner("Searching for jobs using your profile..."):

            structured, agent_error = _run_agent(auto_prompt)
            if agent_error:
                st.warning(agent_error)

            if structured is not None:
                site_name = structured.site or sites
                search_term = structured.search_term or job_title
                google_search_term = structured.google_search_term or f"{job_title} jobs {location}"
                loc = structured.location or location
                h_old = structured.hours_old or hours_old
                results_wanted = structured.results_wanted or 20
                country_indeed = structured.country_indeed or "India"
            else:
                site_name = sites
                search_term = job_title
                google_search_term = f"{job_title} jobs {location} posted in last {hours_old} hours"
                loc = location
                h_old = hours_old
                results_wanted = 20
                country_indeed = "India"

            site_name = _normalize_sites(site_name)
            site_name, unsupported_sites = _ensure_requested_sites(site_name, auto_prompt)
            if not site_name:
                site_name = ["linkedin", "naukri", "indeed"]

            df, scrape_error, site_counts = _run_scrape(
                site_name=site_name,
                search_term=search_term,
                google_search_term=google_search_term,
                location=loc,
                results_wanted=results_wanted,
                hours_old=h_old,
                country_indeed=country_indeed,
                linkedin_fetch_description=True,
            )

            if scrape_error:
                st.error(scrape_error)
            else:
                st.session_state["jobs_df"] = df
                st.session_state["site_counts"] = site_counts
                st.session_state["unsupported_sites"] = unsupported_sites

    st.divider()

else:
    st.warning("⚠️ No saved profile found. Fill in **Your Preferences** to enable profile-based search.")
    if st.button("→ Go to Your Preferences", use_container_width=True):
        st.switch_page("pages/pref.py")
    st.write("")

# =========================================
# MANUAL SEARCH BOX
# =========================================

st.subheader("🔎 Or Search Manually")

prompt = st.text_input(
    "Enter your job request",
    placeholder="Example: I want Python developer jobs in Bengaluru from LinkedIn and Naukri posted in the last 3 days"
)

with st.expander("⚙️ Advanced options (optional)"):
    adv_col1, adv_col2 = st.columns(2)
    with adv_col1:
        manual_results_wanted = st.slider("Results per site", min_value=5, max_value=50, value=20, step=5)
    with adv_col2:
        manual_hours_old = st.slider("Max posting age (hours)", min_value=1, max_value=336, value=72, step=1)

# =========================================
# MANUAL SEARCH BUTTON
# =========================================

if st.button("Search Jobs", use_container_width=True):

    if not prompt:
        st.error("Please enter a job search query.")
    else:
        with st.spinner("Searching for jobs..."):

            structured, agent_error = _run_agent(prompt)
            if agent_error:
                st.warning(agent_error)

            if structured is not None:
                site_name = structured.site or ["indeed", "linkedin", "google", "glassdoor", "naukri"]
                search_term = structured.search_term or "engineer"
                google_search_term = structured.google_search_term or "engineer jobs"
                location = structured.location or "Bengaluru, India"
                hours_old = structured.hours_old or manual_hours_old
                results_wanted = structured.results_wanted or manual_results_wanted
                country_indeed = structured.country_indeed or "India"
            else:
                site_name = ["indeed", "linkedin", "google", "glassdoor", "naukri"]
                search_term = prompt
                google_search_term = prompt
                location = "Bengaluru, India"
                hours_old = manual_hours_old
                results_wanted = manual_results_wanted
                country_indeed = "India"

            site_name = _normalize_sites(site_name)
            site_name, unsupported_sites = _ensure_requested_sites(site_name, prompt)
            if not site_name:
                site_name = ["indeed", "linkedin", "google", "glassdoor", "naukri"]

            df, scrape_error, site_counts = _run_scrape(
                site_name=site_name,
                search_term=search_term,
                google_search_term=google_search_term,
                location=location,
                results_wanted=results_wanted,
                hours_old=hours_old,
                country_indeed=country_indeed,
                linkedin_fetch_description=True,
            )

            if scrape_error:
                st.error(scrape_error)
            else:
                st.session_state["jobs_df"] = df
                st.session_state["site_counts"] = site_counts
                st.session_state["unsupported_sites"] = unsupported_sites

# =========================================
# RENDER RESULTS (persisted across filter changes)
# =========================================

if "jobs_df" in st.session_state:
    st.divider()
    st.subheader("📋 Results")

    unsupported_sites = st.session_state.get("unsupported_sites", [])
    if unsupported_sites:
        st.warning(
            "⚠️ You mentioned **" + ", ".join(s.title() for s in unsupported_sites) +
            "**, but the scraping library this app uses (`jobspy`) doesn't "
            "support that site, so it couldn't be included. Supported sites "
            "are: LinkedIn, Indeed, Glassdoor, Google Jobs, Naukri, "
            "ZipRecruiter, Bayt, and BDJobs."
        )

    site_counts = st.session_state.get("site_counts", {})
    if site_counts:
        zero_sites = [s for s, c in site_counts.items() if c == 0]
        with st.expander("🔍 Search diagnostics (results per site)"):
            st.write({s: c for s, c in site_counts.items()})
            if zero_sites:
                st.caption(
                    "Sites above with 0 results returned nothing for this "
                    "search/time window — this is usually the scraper "
                    "itself (rate limiting, layout changes, or the site "
                    "blocking automated requests), not your search terms."
                )
        if "naukri" in zero_sites:
            st.info(
                "ℹ️ **Naukri returned 0 results.** Naukri's scraper in "
                "`jobspy` is the least stable of the supported sites and "
                "frequently rate-limits or blocks requests (especially "
                "from cloud/Colab IPs) or fails when `hours_old` is set. "
                "Try: removing the time filter, searching again after a "
                "minute, or running from a different network. If it keeps "
                "returning 0 across many searches, it's likely a "
                "`jobspy`/Naukri-side issue rather than something in this app."
            )

    try:
        _render_results(st.session_state["jobs_df"])
    except Exception:
        st.error("Something went wrong while displaying results.")
        st.code(traceback.format_exc())

Writing pages/ai_job_search.py


In [6]:
%%writefile home.py
from bs4 import BeautifulSoup
import streamlit as st
from PIL import Image
import base64
from io import BytesIO

st.set_page_config(layout="wide")

img = Image.open("uimage.jpg").convert("RGBA")
st.markdown('<h6 align="right"> Profile  </h6>', unsafe_allow_html=True)
# Convert image to base64
buffered = BytesIO()
img.save(buffered, format="PNG")
img_str = base64.b64encode(buffered.getvalue()).decode()


st.markdown("""


""", unsafe_allow_html=True)

st.markdown("""
<style>
.faded-image {
    width: 100%;
    height: auto;
    opacity: 0.5;
    -webkit-mask-image: radial-gradient(circle, black 50%, transparent 100%);
    mask-image: radial-gradient(circle, black 50%, transparent 100%);
}
</style>
""", unsafe_allow_html=True)

st.markdown(f"""
<style>



.image-container {{
    position: relative;
    width: 100%;
}}

.faded-image {{
    width: 100%;
    height: auto;
    opacity: 0.5;
    -webkit-mask-image: radial-gradient(circle, black 50%, transparent 100%);
    mask-image: radial-gradient(circle, black 50%, transparent 100%);
}}

.large-text {{
    position: absolute;
    top: 50%;
    left: 50%;
    transform: translate(-50%, -50%);
    font-size: 100px;
    font-weight: bold;
    color: white;
    text-shadow: 3px 3px 20px rgba(0,0,0,0.8);
}}

.medium-text {{

       position: absolute;
    top: 70%;
    left: 50%;
    transform: translate(-50%, -50%);
    font-size: 30px;
    font-weight: bold;
    color: white;
    text-shadow: 3px 3px 20px rgba(0,0,0,0.8);
}}
</style>

<div class="image-container">
    <img src="data:image/png;base64,{img_str}" class="faded-image">
    <div class="large-text">JOBIFY<br></div>
    <div class="medium-text">Your AI Job-Search Assistant</div>
</div>
""", unsafe_allow_html=True)

st.markdown('<br><br><br>', unsafe_allow_html=True)


st.markdown("""
<style>
.rcard {
    width: 300px;
    height: 350px;
    padding: 20px;
    border-radius: 10px;
    background: linear-gradient(135deg, #38069c, #62069c);
    color: white;
    transition: transform 0.3s ease, box-shadow 0.3s ease;
}

.pjcard {
    width: 300px;
    height: 350px;
    padding: 20px;
    border-radius: 10px;
    background: linear-gradient(135deg,#38069c, #147350);
    color: white;
    transition: transform 0.3s ease, box-shadow 0.3s ease;
}

.rcard:hover{
    transform: translateY(-10px);
    box-shadow: 5px 25px 25px rgba(220, 25, 250,0.5);
}

.pjcard:hover {
    transform: translateY(-10px);
    box-shadow: 5px 25px 25px rgba(60, 240, 171,0.5);
}

.ajcard {
    width: 300px;
    height: 350px;
    padding: 20px;
    border-radius: 10px;
    background: linear-gradient(135deg,#38069c, #0f8a88);
    color: white;
    transition: transform 0.3s ease, box-shadow 0.3s ease;
}

.ajcard:hover {
    transform: translateY(-10px);
    box-shadow: 5px 25px 25px rgba(60, 198, 240,0.5);
}

.card-container {
    display: flex;
    justify-content: center;
    gap: 100px;
    margin-top: 50px;
}
</style>
""", unsafe_allow_html=True)


st.markdown("""
<p style="font-size: 18px; font-family :system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, 'Open Sans', 'Helvetica Neue', sans-serif;">
            This AI Job Search Assistant is designed to revolutionize your job hunting experience. By leveraging advanced AI algorithms, we analyze your resume and preferences to curate a personalized list of job opportunities that align with your skills and aspirations. Whether you're looking for specific job titles, locations, or industries, our assistant ensures you receive tailored recommendations to help you find the perfect job match.</p>
""", unsafe_allow_html=True)

resume_img = Image.open("resume.png").convert("RGBA")
buffered_resume = BytesIO()
resume_img.save(buffered_resume, format="PNG")
resume_str = base64.b64encode(buffered_resume.getvalue()).decode()

choiceimg = Image.open("choicetr.png").convert("RGBA")
buffered_choice = BytesIO()
choiceimg.save(buffered_choice, format="PNG")
choice_str = base64.b64encode(buffered_choice.getvalue()).decode()

jobimg = Image.open("jobtr.png").convert("RGBA")
buffered_job = BytesIO()
jobimg.save(buffered_job, format="PNG")
job_str = base64.b64encode(buffered_job.getvalue()).decode()

html_cards = f"""
<div class="card-container">
    <a href="/resume_reader" target="_self" style="text-decoration:none;">
    <div class="rcard">
        <h3 align="center">Upload Resume  ✦</h3>
        <img src="data:image/png;base64,{resume_str}" width="100" style="margin-left: 80px;">
        <p>Simply upoad you Resume and out AI agent will analyse your skills
        to find the best Job Opportunities for you.</p>
    </div>
    </a>
    <a href="/pref" target="_self" style="text-decoration:none;">
    <div class="pjcard">
        <h3 align="center">Your Preferences</h3>
        <img src="data:image/png;base64,{choice_str}" width="150" style="margin-left: 50px;margin-top: 5px;">
        <p> Select you preferred job titles, locations, and industries to get personalized job recommendations.</p>
    </div>
    </a>
    <a href="/ai_job_search" target="_self" style="text-decoration:none;">
      <div class="ajcard">
        <h3 align="center">AI Job Search  ✦</h3>
        <img src="data:image/png;base64,{job_str}" width="125" style="margin-left: 75px;margin-top: 5px;">
        <p>Our AI Agent curates a Job List as per you preferences and resume</p>
      </div>
    </a>
</div>
"""

st.markdown(html_cards, unsafe_allow_html=True)


Writing home.py


In [9]:
from pyngrok import ngrok
import subprocess
from google.colab import userdata

ng=userdata.get('NGROK')
ngrok.set_auth_token(ng)

process = subprocess.Popen(["streamlit", "run", "home.py"])

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://brant-correctional-perceptually.ngrok-free.dev" -> "http://localhost:8501"


In [8]:
from pyngrok import ngrok
ngrok.kill()